Начнём с подготовки данных с помощью Pandas:

Очистим данные от выбросов - более 1000 покупок в день - это могут быть боты. Оставляем в датафрейме только тех пользователей, которые не являются ботами.

In [1]:
import pandas as pd

df = pd.read_csv('2019-Nov.csv')

df['event_time'] = pd.to_datetime(df['event_time'])
df['date'] = df['event_time'].dt.date


purchases = df[df['event_type'] == 'purchase']


daily_purchases = purchases.groupby(['user_id', 'date']).size().reset_index(name='purchase_count')

bot_users = daily_purchases[daily_purchases['purchase_count'] > 1000]['user_id'].unique()

df_filtered = df[~df['user_id'].isin(bot_users)].copy()

df_filtered = df_filtered.drop(columns=['date'])

Нам понадобятся только целевые события view -> cart -> purchase - то есть только те товары, которые положили в корзину и затем купили. Дополнительно зачистим артефакты, которые были куплены, но предварительно не попали в корзину.

In [2]:
events_to_keep = ['view', 'cart', 'purchase']
df_funnel = df_filtered[df_filtered['event_type'].isin(events_to_keep)].copy()

df_funnel = df_funnel.sort_values(by=['user_session', 'product_id', 'event_time'])

df_funnel['is_cart'] = (df_funnel['event_type'] == 'cart').astype(int)

df_funnel['cart_cumsum'] = df_funnel.groupby(['user_session', 'product_id'])['is_cart'].cumsum()

anomaly_purchases_mask = (df_funnel['event_type'] == 'purchase') & (df_funnel['cart_cumsum'] == 0)

df_clean = df_funnel[~anomaly_purchases_mask].drop(columns=['is_cart', 'cart_cumsum'])

df_clean = df_clean.sort_values(by='event_time').reset_index(drop=True)

Теперь рассчитаем макро-конверсию ((Overall CR): % дошедших до покупки от зашедших в приложение) и микро-конверсии ((Step-to-step CR): % перехода из просмотра в корзину и из корзины в оплату.)

In [3]:
funnel_stages = df_clean.groupby('event_type')['user_id'].nunique().reindex(['view', 'cart', 'purchase'])

users_view = funnel_stages['view']
users_cart = funnel_stages['cart']
users_purchase = funnel_stages['purchase']

macro_cr = (users_purchase / users_view) * 100

cr_view_to_cart = (users_cart / users_view) * 100
cr_cart_to_purchase = (users_purchase / users_cart) * 100

metrics_df = pd.DataFrame({
    'Stage': ['view', 'cart', 'purchase'],
    'Unique_Users': [users_view, users_cart, users_purchase],
    'Step_CR_%': [100.0, cr_view_to_cart, cr_cart_to_purchase],
    'Overall_CR_%': [100.0, (users_cart / users_view) * 100, macro_cr]
})

print(metrics_df.round(2))

      Stage  Unique_Users  Step_CR_%  Overall_CR_%
0      view        355643     100.00        100.00
1      cart         95939      26.98         26.98
2  purchase         25571      26.65          7.19


По данному выводу визуализируем воронку

In [4]:
import plotly.graph_objects as go

stages_labels = ['Просмотр - View', 'Корзина - Cart', 'Покупка - Purchase']

fig = go.Figure(go.Funnel(
    y=stages_labels,
    x=metrics_df['Unique_Users'],
    textposition="inside",
    textinfo="value+percent previous+percent initial",
    opacity=0.85,
    marker={
        "color": ["#4C78A8", "#F58518", "#54A24B"],
        "line": {"width": [2, 2, 2], "color": ["white", "white", "white"]}
    },
    connector={"line": {"color": "royalblue", "dash": "dot", "width": 2}}
))

fig.update_layout(
    title="Воронка конверсии мобильного приложения",
    title_x=0.5, 
    template="plotly_white",
    width=800,
    height=500
)

fig.show()

In [5]:
fig.write_image("funnel_chart.png", scale=2)